In [0]:
%restart_python

In [0]:
from src.transforms.gold import build_dim_date, latest_actor_state, event_type_dim

CATALOG = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

events = spark.table(f"{CATALOG}.silver.events")

build_dim_date(spark, "2026-05-01", "2026-12-31").write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_date")

latest_actor_state(events).write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_actor")
event_type_dim(events).write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_event_type")
print("dim built")


In [0]:
spark.sql(f"SELECT count(*) AS days FROM {CATALOG}.gold.dim_date").show()

spark.sql(f"SELECT count(*) AS actors, sum(cast(is_bot AS int)) AS bots FROM {CATALOG}.gold.dim_actor").show()
spark.sql(f"SELECT * FROM {CATALOG}.gold.dim_event_type ORDER BY event_type").show(30)
spark.sql(f"""
          SELECT date_key, date, day_name, is_weekend
          FROM {CATALOG}.gold.dim_date WHERE date = '2026-06-01'
          """).show()